# Experiment Group 4: Initialization Quality

**Thesis:** Cell-Type Dynamical Systems (CTDS) — Biologically Constrained Linear Dynamical Systems  
**Chapter:** 6 — Model Validation

---

**Purpose:** Demonstrate that the NMF-based initialization is substantially better than random initialization and understand why. EM for CTDS is a non-convex optimization with many local optima. The choice of starting point materially affects both the speed of convergence and the quality of the final solution. This experiment group quantifies that effect rigorously.

**Experiments in this group:**
- **4.1** — NMF vs. random initialization: LL traces, recovery error at convergence, convergence speed.
- **4.2** — NMF advantage vs. data volume: does the gap shrink as T·B grows?
- **4.3** — Quality of NMF initialization itself, before any EM iterations (three-stage comparison).

**Governing principle:** Every experiment states its question, its hypothesis, its protocol, and its expected outcome before any code runs. All parameter comparisons use the `transform_true_rec` alignment to correct for within-block permutation and diagonal scaling gauge freedom.

## Section 0: Setup, Imports, and Shared Helpers

In [ ]:
# EXPLANATION: 64-bit floats are mandatory. JAX defaults to 32-bit, which
# causes numerical instability in the Kalman smoother and the BoxCDQP solver
# used in the constrained M-step. This line must precede all JAX imports.
import jax
jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import jax.random as jr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import seaborn as sns
from functools import partial


from ctds.params import (
    ParamsCTDS, ParamsCTDSInitial, ParamsCTDSDynamics,
    ParamsCTDSEmissions, ParamsCTDSConstraints
)
from ctds.models import CTDS
from ctds.simulation_utils import generate_synthetic_data, transform_true_rec, transform_true_rec_hungarian

print(f"JAX version        : {jax.__version__}")
print(f"JAX backend        : {jax.default_backend()}")
print(f"64-bit enabled     : {jax.config.jax_enable_x64}")

#configs to save figures
import matplotlib.pyplot as _plt_module
import os
# Only patch if not already patched
if not getattr(_plt_module, '_savefig_patched', False):
    _EXP0_FIG_DIR = os.path.join("figures", "exp4", "rand_versus_nmf")
    os.makedirs(_EXP0_FIG_DIR, exist_ok=True)
    _orig_savefig = plt.savefig

    def _routed_savefig(fname, **kwargs):
        _orig_savefig(os.path.join(_EXP0_FIG_DIR, os.path.basename(str(fname))), **kwargs)

    plt.savefig = _routed_savefig
    _plt_module._savefig_patched = True
    print("Auto-save patched → figures/exp2_recovery/")
else:
    print("Auto-save already active (skipping re-patch).")

In [ ]:
# ── Global matplotlib style ───────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi'             : 120,
    'font.size'              : 11,
    'axes.spines.top'        : False,
    'axes.spines.right'      : False,
    'axes.labelsize'         : 11,
    'axes.titlesize'         : 13,
    'xtick.labelsize'        : 9,
    'ytick.labelsize'        : 9,
    'legend.fontsize'        : 9,
    'figure.constrained_layout.use': True,
})
PALETTE   = sns.color_palette("colorblind")   # 10-colour colourblind-safe palette
GRAY_THIN = dict(color='#AAAAAA', linewidth=0.8, alpha=0.6)   # style for random-init traces

# ── Global random key ─────────────────────────────────────────────────────────
KEY_BASE = jr.PRNGKey(42)

# ── Default experimental configuration ───────────────────────────────────────
D_DEFAULT    = 4      # total latent dimensions (4E + 4I)
N_E_DEFAULT  = 10     # excitatory neurons
N_I_DEFAULT  = 10     # inhibitory neurons
N_DEFAULT    = 20     # total neurons
T_DEFAULT    = 500    # time steps per trial
B_DEFAULT    = 10     # number of training trials
K            = 2      # number of cell types
NUM_SEEDS    = 10     # seeds per condition
NUM_EM_ITERS = 200    # EM iterations (longer than Group 2 to see full convergence)

# Iteration checkpoints for Experiment 4.1 convergence comparison
ITER_CHECKPOINTS = [10, 25, 50, 100, 200]

print("Default configuration:")
print(f"  D={D_DEFAULT} ({D_DEFAULT//2}E + {D_DEFAULT - D_DEFAULT//2}I)")
print(f"  N={N_DEFAULT} ({N_E_DEFAULT}E + {N_I_DEFAULT}I)")
print(f"  T={T_DEFAULT}, B={B_DEFAULT}")
print(f"  EM iterations : {NUM_EM_ITERS}")
print(f"  Seeds/condition: {NUM_SEEDS}")

In [ ]:
# ── Shared metric helpers ─────────────────────────────────────────────────────

def relative_frob_error(M_rec, M_true):
    """||M_rec - M_true||_F / ||M_true||_F  — returns a Python float."""
    num = jnp.linalg.norm(M_rec - M_true, 'fro')
    den = jnp.linalg.norm(M_true,         'fro')
    return float(num / (den + 1e-12))


def align_and_compute_errors(true_params, rec_params):
    """
    Apply transform_true_rec alignment then return a dict of all error metrics.

    EXPLANATION: transform_true_rec removes the residual gauge freedom
    characterised in Chapter 5 — within-block permutation ambiguity and positive
    diagonal rescaling. Computing errors on unaligned parameters would give
    misleadingly large values even when the model has learned the correct dynamics.

    Returns
    -------
    dict with keys: epsilon_A, epsilon_C, epsilon_Q, epsilon_R,
                    _A_rec_al, _C_rec_al, _Q_rec_al, _R_rec  (for plotting)
    """
    constraints = true_params.constraints
    cell_dims   = np.array(constraints.cell_type_dimensions)   # shape (K,)
    list_of_dims = cell_dims[np.newaxis, :]                    # shape (1, K)

    C_true = np.array(true_params.emissions.weights)
    A_true = np.array(true_params.dynamics.weights)
    Q_true = np.array(true_params.dynamics.cov)
    R_true = np.array(true_params.emissions.cov)

    C_rec  = np.array(rec_params.emissions.weights)
    A_rec  = np.array(rec_params.dynamics.weights)
    Q_rec  = np.array(rec_params.dynamics.cov)
    R_rec  = np.array(rec_params.emissions.cov)

    dic = transform_true_rec_hungarian( C_true, C_rec, A_rec, Q_rec, list_of_dims, region_identity=None)
    if dic['collapsed']:
        # Fall back to unaligned error (will be large, but won't crash)
        A_rec_al, C_rec_al, Q_rec_al = A_rec, C_rec, Q_rec
    else:
        A_rec_al, C_rec_al, Q_rec_al = dic['A_aligned'], dic['C_aligned'], dic['Q_aligned']
    return {
        'epsilon_A' : relative_frob_error(A_rec_al, A_true),
        'epsilon_C' : relative_frob_error(C_rec_al, C_true),
        'epsilon_Q' : relative_frob_error(Q_rec_al, Q_true),
        'epsilon_R' : relative_frob_error(R_rec,    R_true),
        '_A_rec_al' : A_rec_al,
        '_C_rec_al' : C_rec_al,
        '_Q_rec_al' : Q_rec_al,
        '_R_rec'    : R_rec,
    }


def save_figure(fig, name):
    """Save figure as PNG (300 dpi) and PDF."""
    fig.savefig(f"{name}.png", dpi=300, bbox_inches='tight')
    print(f"  Saved: {name}.png  +  {name}.pdf")


print("Helpers defined: relative_frob_error, align_and_compute_errors, save_figure")

In [ ]:
# ── Core data-generation helper ───────────────────────────────────────────────

def generate_ground_truth_and_data(
    seed_key,
    T=T_DEFAULT, B=B_DEFAULT,
    D=D_DEFAULT, N_E=N_E_DEFAULT, N_I=N_I_DEFAULT
):
    """
    Generate ground-truth CTDS parameters and B training trials.

    Returns
    -------
    ctds_model        : instantiated CTDS object
    true_params       : ParamsCTDS ground-truth
    batch_observations: jnp.array shape (B, T, N)
    batch_states      : jnp.array shape (B, T, D)
    """
    N = N_E + N_I
    key_gen, key_trials = jr.split(seed_key, 2)

    states, obs, ctds_model, true_params = generate_synthetic_data(
        num_samples=B, num_timesteps=T,
        state_dim=D, emission_dim=N,
        cell_types=K, key=key_gen
    )
    return ctds_model, true_params, obs, states



print("generate_ground_truth_and_data defined.")

In [ ]:
# ── Manual EM loop (records per-iteration LL and per-checkpoint errors) ────────
#
# EXPLANATION: We implement the EM loop manually rather than calling fit_em
# because:
#   (a) We need the log-likelihood after EVERY iteration for trace plots.
#   (b) We need the parameter state at specific iteration checkpoints
#       (10, 25, 50, 100) to compare recovery error at fixed budgets.
#   fit_em only returns the final parameters and the LL at each iteration,
#   so (b) requires a manual loop.

def run_em_manual(
    ctds_model,
    init_params,
    batch_observations,
    true_params,
    num_iters=NUM_EM_ITERS,
    checkpoints=None
):
    """
    Run EM manually, recording LL at every iteration and errors at checkpoints.

    Parameters
    ----------
    ctds_model        : CTDS model instance.
    init_params       : ParamsCTDS starting point.
    batch_observations: (B, T, N) observations.
    true_params       : ground-truth ParamsCTDS for error computation.
    num_iters         : total EM iterations.
    checkpoints       : list of iteration indices at which to record errors.
                        If None, only the final iteration is recorded.

    Returns
    -------
    dict with:
        ll_trace         : list of float, length num_iters+1  (LL after init E-step,
                           then after each M-step)
        checkpoint_errors: dict {iter_idx: error_dict} for each checkpoint
        final_params     : ParamsCTDS after num_iters
        final_errors     : error dict at convergence
    """
    if checkpoints is None:
        checkpoints = [num_iters]

    B, T, _ = batch_observations.shape
    m_step_state = ctds_model.initialize_m_step_state(init_params, num_iters)

    params       = init_params
    ll_trace     = []
    checkpoint_errors = {}

    for i in range(num_iters):
        # E-step: vmap over trials
        # EXPLANATION: each trial is independent given params, so we vmap the
        # e_step over the batch dimension.  The sufficient statistics are then
        # summed across trials in the M-step.
        batch_stats, lls = jax.vmap(
            partial(ctds_model.e_step, params)
        )(batch_observations)

        ll_trace.append(float(jnp.sum(lls)))

        # M-step
        params, m_step_state = ctds_model.m_step(
            params, None, batch_stats, m_step_state
        )

        # Record errors at checkpoints (1-indexed: checkpoint k means after k iters)
        if (i + 1) in checkpoints:
            checkpoint_errors[i + 1] = align_and_compute_errors(true_params, params)

    # Final E-step LL (after last M-step)
    _, final_lls = jax.vmap(
        partial(ctds_model.e_step, params)
    )(batch_observations)
    ll_trace.append(float(jnp.sum(final_lls)))

    final_errors = align_and_compute_errors(true_params, params)

    return {
        'll_trace'         : ll_trace,           # length num_iters + 1
        'checkpoint_errors': checkpoint_errors,
        'final_params'     : params,
        'final_errors'     : final_errors,
    }


print("run_em_manual defined.")

In [ ]:
# ── Initialization helpers ─────────────────────────────────────────────────────
#
# CTDS.initialize() wraps the NMF-based initialization strategy.
# For random initialization we use the same API but pass a randomly perturbed
# observation to break any data-driven structure.
# EXPLANATION: a truly random init (e.g. iid Gaussian entries in A and C) would
# almost certainly violate the Dale's law constraints.  Instead we call
# ctds_model.initialize() on a noise-only tensor, which uses the same NMF
# code path but with structureless input — giving a random-quality starting
# point that still satisfies all constraints.

from ctds.simulation_utils import generate_random_CTDS_params


def get_nmf_init(ctds_model, batch_observations):
    """Return NMF-based initial parameters from the first training trial."""
    return ctds_model.initialize(batch_observations)

def get_random_init(ctds_model, params_true, batch_observations, key):
    B, T, N = batch_observations.shape
    D,_= params_true.dynamics.weights.shape
    """
    k1, k2, k3, k4 = jr.split(key, 4)
    C_rand = jnp.abs(jr.normal(k1, (N, D))) 
    A_rand = jr.normal(k2, (D, D)) * 0.1
    Q_rand = jax.random.normal(k3, (D, D))
    R_rand = jnp.diag(jnp.abs(jr.normal(k4, (N,))))
    return ParamsCTDS(
        initial=ParamsCTDSInitial(mean=jnp.zeros(D), cov=jnp.eye(D)),
        dynamics=ParamsCTDSDynamics(weights=A_rand, cov=Q_rand, dynamics_mask=params_true.dynamics.dynamics_mask),
        emissions=ParamsCTDSEmissions(weights=C_rand, cov=R_rand, bias=jnp.zeros(N)),
        constraints=params_true.constraints,
        observations=batch_observations.reshape(-1),)
    """
    params=generate_random_CTDS_params(N,T,D, seed=key)
    params = params._replace(constraints=params_true.constraints, observations=batch_observations)
    return params
"""
def get_random_init(ctds_model, batch_observations, key):
    B, T, N = batch_observations.shape
    noise_obs = jr.normal(key, shape=(B,T ,N))
    # Scale noise to have similar variance to the data (prevents degenerate init)
    data_std  = jnp.std(batch_observations)
    noise_obs = noise_obs * data_std
    print(noise_obs.shape)
    return ctds_model.initialize(noise_obs)
"""

print("get_nmf_init, get_random_init defined.")

---
## Section 1: Preliminary — Verify Initialization Outputs

Before running the full experiments, we verify that:
1. `get_nmf_init` and `get_random_init` return valid `ParamsCTDS` objects.
2. Both satisfy Dale's law and non-negativity constraints.
3. We can compute an initial error metric from each — establishing a baseline.

In [ ]:
print("Preliminary: verifying initialization outputs...")

key_prelim = jr.PRNGKey(99)
ctds_pre, tp_pre, obs_pre, _ = generate_ground_truth_and_data(
    key_prelim, T=T_DEFAULT, B=B_DEFAULT,
    D=D_DEFAULT, N_E=N_E_DEFAULT, N_I=N_I_DEFAULT
)

nmf_init    = get_nmf_init(ctds_pre, obs_pre)
rand_init   = get_random_init(ctds_pre,tp_pre,obs_pre, jr.PRNGKey(0))

errs_nmf_pre  = align_and_compute_errors(tp_pre, nmf_init)
errs_rand_pre = align_and_compute_errors(tp_pre, rand_init)

print("\nNMF init (before any EM):")
print(f"  epsilon_A = {errs_nmf_pre['epsilon_A']:.4f}")
print(f"  epsilon_C = {errs_nmf_pre['epsilon_C']:.4f}")

print("\nRandom init (before any EM):")
print(f"  epsilon_A = {errs_rand_pre['epsilon_A']:.4f}")
print(f"  epsilon_C = {errs_rand_pre['epsilon_C']:.4f}")

# Constraint checks
def check_constraints(params, label):
    dyn_mask = np.array(params.dynamics.dynamics_mask)   # +1 / -1 per latent dim
    A = np.array(params.dynamics.weights)
    D = A.shape[0]
    dale_viol = 0.0
    for j in range(D):
        for i in range(D):
            if i != j:
                dale_viol = max(dale_viol, max(0.0, -dyn_mask[j] * A[i, j]))
    C = np.array(params.emissions.weights)
    N_E_loc = int(jnp.sum(params.constraints.cell_type_mask == 0))
    D_E_loc = int(params.constraints.cell_type_dimensions[0])
    nonneg_viol   = float(np.max(np.maximum(0, -C[C > -1e-12])))
    struct_zero   = float(np.max(np.abs(C[N_E_loc:, :D_E_loc])))
    q_min_eig     = float(jnp.min(jnp.linalg.eigvalsh(params.dynamics.cov)))
    print(f"\n  {label} constraint check:")
    print(f"    Dale violation in A:      {dale_viol:.2e}  (should be ~0)")
    print(f"    Non-neg violation in C:   {nonneg_viol:.2e}  (should be ~0)")
    print(f"    Struct-zero violation C:  {struct_zero:.2e}  (should be ~0)")
    print(f"    Q min eigenvalue:         {q_min_eig:.4f}   (should be > 0)")

check_constraints(nmf_init,  "NMF init")
check_constraints(rand_init, "Random init")
print("\nPreliminary checks complete.")

---
## Section 2: Experiment 4.1 — NMF vs. Random Initialization

**Question:** Does the NMF-based initialization lead to better parameter recovery and faster convergence than random initialization?

**Hypothesis:** EM started from the NMF initialization converges to lower recovery error and higher log-likelihood than EM from random initialization, in fewer iterations. Specifically:
- NMF traces start higher on the LL curve and remain higher throughout.
- At convergence (200 iterations), NMF achieves lower ε_A and ε_C with smaller variance across seeds.
- NMF reaches its final ε_A in fewer iterations than random (faster convergence speed).

**Protocol:**
- 10 seeds × 2 strategies (NMF, random) at the default configuration.
- 200 EM iterations per run, recording LL at every iteration.
- Errors recorded at checkpoints: 10, 25, 50, 100, 200 iterations.
- Three plots: (4.1a) LL traces, (4.1b) box plots at convergence, (4.1c) convergence speed.

In [ ]:
print("Experiment 4.1: NMF vs. Random Initialization")
print(f"  {NUM_SEEDS} seeds × 2 strategies × {NUM_EM_ITERS} EM iterations")
print(f"  Checkpoints: {ITER_CHECKPOINTS}")
print("  (This is the most compute-intensive cell — expect ~10–20 min on CPU)\n")

seed_keys_41 = jr.split(jr.PRNGKey(41), NUM_SEEDS)

# Storage: list of result dicts, one per seed per strategy
results_41_nmf  = []   # NMF initialization
results_41_rand = []   # Random initialization

for i, sk in enumerate(seed_keys_41):
    key_data, key_rand_init = jr.split(sk, 2)

    # ── Generate shared ground truth for this seed ─────────────────────────
    ctds_41, tp_41, obs_41, _ = generate_ground_truth_and_data(
        key_data,
        T=T_DEFAULT, B=B_DEFAULT,
        D=D_DEFAULT, N_E=N_E_DEFAULT, N_I=N_I_DEFAULT
    )
    # EXPLANATION: Both strategies see the same ground-truth params and the
    # same observations. The only difference is the starting point for EM.

    # ── NMF initialization ─────────────────────────────────────────────────
    nmf_ip = get_nmf_init(ctds_41, obs_41)
    res_nmf = run_em_manual(
        ctds_41, nmf_ip, obs_41, tp_41,
        num_iters=NUM_EM_ITERS,
        checkpoints=ITER_CHECKPOINTS
    )
    results_41_nmf.append(res_nmf)

    # ── Random initialization ──────────────────────────────────────────────
    rand_ip = get_random_init(ctds_41,tp_41, obs_41, key_rand_init)
    res_rand = run_em_manual(
        ctds_41, rand_ip, obs_41, tp_41,
        num_iters=NUM_EM_ITERS,
        checkpoints=ITER_CHECKPOINTS
    )
    results_41_rand.append(res_rand)

    print(f"  Seed {i+1:2d}/{NUM_SEEDS}  "
          f"NMF  final: LL={res_nmf['ll_trace'][-1]:9.2f}, "
          f"ε_A={res_nmf['final_errors']['epsilon_A']:.4f}  |  "
          f"RAND final: LL={res_rand['ll_trace'][-1]:9.2f}, "
          f"ε_A={res_rand['final_errors']['epsilon_A']:.4f}",
          flush=True)

print("\nExperiment 4.1 data collection complete.")

In [ ]:
# ── Extract arrays for plotting ────────────────────────────────────────────────

# LL traces: shape (NUM_SEEDS, NUM_EM_ITERS+1)
ll_nmf_mat  = np.array([r['ll_trace'] for r in results_41_nmf])
ll_rand_mat = np.array([r['ll_trace'] for r in results_41_rand])
iters_axis  = np.arange(NUM_EM_ITERS + 1)  # 0, 1, ..., 200

# LL medians
ll_nmf_med  = np.median(ll_nmf_mat,  axis=0)
ll_rand_med = np.median(ll_rand_mat, axis=0)

# Final errors across seeds
eA_nmf_final  = np.array([r['final_errors']['epsilon_A'] for r in results_41_nmf])
eC_nmf_final  = np.array([r['final_errors']['epsilon_C'] for r in results_41_nmf])
eA_rand_final = np.array([r['final_errors']['epsilon_A'] for r in results_41_rand])
eC_rand_final = np.array([r['final_errors']['epsilon_C'] for r in results_41_rand])

# Checkpoint errors: epsilon_A at each checkpoint for each seed
# Shape: (NUM_SEEDS, len(ITER_CHECKPOINTS))
def extract_checkpoint_eA(results_list, checkpoints):
    mat = np.zeros((len(results_list), len(checkpoints)))
    for i, res in enumerate(results_list):
        for j, ck in enumerate(checkpoints):
            mat[i, j] = res['checkpoint_errors'][ck]['epsilon_A']
    return mat

ck_eA_nmf  = extract_checkpoint_eA(results_41_nmf,  ITER_CHECKPOINTS)
ck_eA_rand = extract_checkpoint_eA(results_41_rand, ITER_CHECKPOINTS)

# Summary statistics
print("Experiment 4.1 — Summary statistics at convergence (200 iters):")
print(f"  NMF   ε_A: mean={np.mean(eA_nmf_final):.4f}, median={np.median(eA_nmf_final):.4f}, "
      f"std={np.std(eA_nmf_final):.4f}")
print(f"  RAND  ε_A: mean={np.mean(eA_rand_final):.4f}, median={np.median(eA_rand_final):.4f}, "
      f"std={np.std(eA_rand_final):.4f}")
print(f"  NMF   ε_C: mean={np.mean(eC_nmf_final):.4f}, median={np.median(eC_nmf_final):.4f}")
print(f"  RAND  ε_C: mean={np.mean(eC_rand_final):.4f}, median={np.median(eC_rand_final):.4f}")
print(f"  NMF  final LL (median): {np.median(ll_nmf_mat[:, -1]):.2f}")
print(f"  RAND final LL (median): {np.median(ll_rand_mat[:, -1]):.2f}")

# Sanity assertion
try:
    assert np.median(eA_nmf_final) < np.median(eA_rand_final), \
        (f"NMF median ε_A ({np.median(eA_nmf_final):.4f}) is NOT lower than "
         f"random ({np.median(eA_rand_final):.4f})")
    print("  ASSERTION PASSED: NMF has lower median ε_A than random.")
except AssertionError as e:
    print(f"  WARNING: {e}")

In [ ]:
# ── Plot 4.1.a — Log-likelihood traces: NMF vs. Random ────────────────────────
#
# Design: two overlaid sets of traces in one panel.
# Random traces: thin grey lines. NMF traces: thin coloured lines.
# Bold lines: the group medians, making the central tendency clear.

fig, ax = plt.subplots(figsize=(9, 5))

# Random traces (plot first so NMF sits on top)
for j, trace in enumerate(ll_rand_mat):
    ax.plot(iters_axis, trace,
            label='Random (individual)' if j == 0 else '_nolegend_',
            **GRAY_THIN)

# NMF traces
for j, trace in enumerate(ll_nmf_mat):
    ax.plot(iters_axis, trace,
            color=PALETTE[0], linewidth=0.8, alpha=0.55,
            label='NMF (individual)' if j == 0 else '_nolegend_')

# Medians
ax.plot(iters_axis, ll_rand_med,
        color="#E02121", linewidth=2.2, linestyle='--',
        label='Random median')
ax.plot(iters_axis, ll_nmf_med,
        color=PALETTE[0], linewidth=2.2, linestyle='-',
        label='NMF median')

# Mark iteration-budget checkpoints
for ck in ITER_CHECKPOINTS[:-1]:  # skip 200 (the endpoint)
    ax.axvline(x=ck, color='lightgrey', linewidth=0.8, linestyle=':')
    ax.text(ck + 1, ax.get_ylim()[0] if ax.get_ylim()[0] != 0 else -1,
            str(ck), fontsize=7, color='grey', va='bottom')

ax.set_xlabel('EM Iteration')
ax.set_ylabel('Marginal Log-Likelihood')
ax.set_title(
    'Experiment 4.1a: Log-Likelihood Traces — NMF vs. Random Initialization\n'
    f'({NUM_SEEDS} seeds, {NUM_EM_ITERS} iterations, default config)'
)
ax.legend(loc='lower right', fontsize=9)
save_figure(fig, 'exp4_1a_ll_traces')
plt.show()

# Verify monotonicity for both strategies
for name, mat in [('NMF', ll_nmf_mat), ('Random', ll_rand_mat)]:
    non_mono = [(seed, it) for seed in range(NUM_SEEDS)
                for it in range(1, NUM_EM_ITERS + 1)
                if mat[seed, it] < mat[seed, it-1] - 1e-6]
    if non_mono:
        print(f"  WARNING: {name} has {len(non_mono)} non-monotone steps: {non_mono[:5]}")
    else:
        print(f"  {name}: all {NUM_SEEDS} traces are monotonically non-decreasing. ✓")

In [ ]:
# ── Plot 4.1.b — Recovery error at convergence: box plots ─────────────────────
#
# Box plots for ε_A, ε_C, ε_Q, ε_R — NMF vs. random.
# Shows full distribution across seeds to reveal variance reduction, not just
# median improvement.

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, (nmf_vals, rand_vals, metric_label) in zip(
    axes,
    [
        (eA_nmf_final, eA_rand_final, r'$\epsilon_A$'),
        (eC_nmf_final, eC_rand_final, r'$\epsilon_C$'),
        #(eQ_nmf_final, eQ_rand_final, r'$\epsilon_Q$'),
        #(eR_nmf_final, eR_rand_final, r'$\epsilon_R$'),
    ]
):
    bp = ax.boxplot(
        [nmf_vals, rand_vals],
        labels=['NMF init', 'Random init'],
        patch_artist=True,
        medianprops=dict(color='black', linewidth=2),
        whiskerprops=dict(linewidth=1.2),
        capprops=dict(linewidth=1.2),
        flierprops=dict(marker='o', markersize=4, alpha=0.5),
        widths=0.4,
    )
    bp['boxes'][0].set_facecolor(PALETTE[0])   # NMF: blue
    bp['boxes'][1].set_facecolor(PALETTE[7])   # Random: grey
    for patch in bp['boxes']:
        patch.set_alpha(0.65)

    # Scatter individual points for transparency
    for x_pos, vals in [(1, nmf_vals), (2, rand_vals)]:
        jitter = np.random.default_rng(0).uniform(-0.08, 0.08, size=len(vals))
        ax.scatter(x_pos + jitter, vals, color='black', s=18, alpha=0.6, zorder=3)

    ax.set_ylabel('Normalised Frobenius Error')
    ax.set_title(f'{metric_label} at convergence ({NUM_EM_ITERS} iters)')

    # Annotate medians
    ax.text(1, np.median(nmf_vals)  * 1.02,
            f'{np.median(nmf_vals):.3f}',  ha='center', fontsize=8, color='black')
    ax.text(2, np.median(rand_vals) * 1.02,
            f'{np.median(rand_vals):.3f}', ha='center', fontsize=8, color='black')

fig.suptitle('Experiment 4.1b: Recovery Error at Convergence — NMF vs. Random', fontsize=13)
save_figure(fig, 'exp4_1b_convergence_boxplots')
plt.show()

In [ ]:
# ── Plot 4.1.c — Convergence speed comparison ─────────────────────────────────
#
# Plot median ε_A vs. iteration (log x-axis) for both strategies.
# The gap between the two curves at any iteration = the "value" of NMF init
# at that compute budget.  If NMF starts near its final error, its curve
# is approximately flat; random decreases steeply before flattening.

# We have ε_A at checkpoint iterations.  Also include the pre-EM error
# (iteration 0) from the preliminary cell above as reference.
# EXPLANATION: We re-compute the iteration-0 errors for all seeds here
# rather than storing them separately in run_em_manual, because recording
# checkpoint_errors at iteration 0 would require a special-case in the loop.

# Compute pre-EM (iter=0) errors for all seeds
eA_nmf_iter0_list  = []
eA_rand_iter0_list = []

for i, sk in enumerate(seed_keys_41):
    key_data, key_rand_init = jr.split(sk, 2)
    ctds_sp, tp_sp, obs_sp, _ = generate_ground_truth_and_data(
        key_data, T=T_DEFAULT, B=B_DEFAULT,
        D=D_DEFAULT, N_E=N_E_DEFAULT, N_I=N_I_DEFAULT
    )
    nmf_ip_sp  = get_nmf_init(ctds_sp, obs_sp)
    rand_ip_sp = get_random_init(ctds_sp, tp_sp,obs_sp, key_rand_init)
    eA_nmf_iter0_list.append(align_and_compute_errors(tp_sp, nmf_ip_sp)['epsilon_A'])
    eA_rand_iter0_list.append(align_and_compute_errors(tp_sp, rand_ip_sp)['epsilon_A'])

eA_nmf_iter0  = np.array(eA_nmf_iter0_list)
eA_rand_iter0 = np.array(eA_rand_iter0_list)

# Build the full iteration axis including 0
full_iters = np.array([0] + ITER_CHECKPOINTS)  # [0, 10, 25, 50, 100, 200]

# Median ε_A at each point
nmf_speed  = np.array(
    [np.median(eA_nmf_iter0)] +
    [np.median(ck_eA_nmf[:, j]) for j in range(len(ITER_CHECKPOINTS))]
)
rand_speed = np.array(
    [np.median(eA_rand_iter0)] +
    [np.median(ck_eA_rand[:, j]) for j in range(len(ITER_CHECKPOINTS))]
)
# 25th–75th percentile bands
nmf_lo  = np.array([np.percentile(eA_nmf_iter0,  25)] +
                   [np.percentile(ck_eA_nmf[:,  j], 25) for j in range(len(ITER_CHECKPOINTS))])
nmf_hi  = np.array([np.percentile(eA_nmf_iter0,  75)] +
                   [np.percentile(ck_eA_nmf[:,  j], 75) for j in range(len(ITER_CHECKPOINTS))])
rand_lo = np.array([np.percentile(eA_rand_iter0, 25)] +
                   [np.percentile(ck_eA_rand[:, j], 25) for j in range(len(ITER_CHECKPOINTS))])
rand_hi = np.array([np.percentile(eA_rand_iter0, 75)] +
                   [np.percentile(ck_eA_rand[:, j], 75) for j in range(len(ITER_CHECKPOINTS))])

fig, ax = plt.subplots(figsize=(7, 4))

# NMF: solid coloured line with IQR band
ax.plot(full_iters, nmf_speed, color=PALETTE[0], linewidth=2.0, linestyle='-',
        marker='o', markersize=5, label='NMF (median ε_A)')
ax.fill_between(full_iters, nmf_lo, nmf_hi, color=PALETTE[0], alpha=0.18,
                label='NMF IQR')

# Random: dashed grey line with IQR band
ax.plot(full_iters, rand_speed, color='#555555', linewidth=2.0, linestyle='--',
        marker='s', markersize=5, label='Random (median ε_A)')
ax.fill_between(full_iters, rand_lo, rand_hi, color='#555555', alpha=0.12,
                label='Random IQR')

ax.set_xscale('log')
# Custom ticks at the checkpoint values
ax.set_xticks(full_iters[1:])  # skip 0 (log(0) = -inf)
ax.set_xticklabels([str(x) for x in full_iters[1:]])
ax.set_xlabel('EM Iteration (log scale)')
ax.set_ylabel(r'Median $\epsilon_A$ across seeds')
ax.set_title(
    'Experiment 4.1c: Convergence Speed — NMF vs. Random Initialization\n'
    f'(Shaded band = IQR across {NUM_SEEDS} seeds)'
)
ax.legend(fontsize=8)

# Mark the pre-EM starting points explicitly
ax.annotate(f'NMF start\n({nmf_speed[0]:.3f})',
            xy=(full_iters[1], nmf_speed[1]),
            xytext=(full_iters[1]*1.3, nmf_speed[1]*1.2),
            fontsize=7, color=PALETTE[0],
            arrowprops=dict(arrowstyle='->', color=PALETTE[0], lw=0.8))
ax.annotate(f'Rand start\n({rand_speed[0]:.3f})',
            xy=(full_iters[1], rand_speed[1]),
            xytext=(full_iters[1]*1.3, rand_speed[1]*0.85),
            fontsize=7, color='#555555',
            arrowprops=dict(arrowstyle='->', color='#555555', lw=0.8))

save_figure(fig, 'exp4_1c_convergence_speed')
plt.show()

print("\nExperiment 4.1 plots complete.")
print(f"  NMF  median ε_A after 10 iters: {nmf_speed[1]:.4f}")
print(f"  RAND median ε_A after 10 iters: {rand_speed[1]:.4f}")
print(f"  NMF  median ε_A after 200 iters: {nmf_speed[-1]:.4f}")
print(f"  RAND median ε_A after 200 iters: {rand_speed[-1]:.4f}")

In [ ]:
# ── Plot 4.1.c — Convergence speed comparison — all four metrics ───────────────
#
# For each metric: median error vs. iteration (log x-axis), IQR band.
# Iter-0 (pre-EM) errors are re-computed for all seeds, as before.

METRICS_41C = [
    ('epsilon_A', r'$\epsilon_A$  (dynamics)'),
    ('epsilon_C', r'$\epsilon_C$  (emissions)'),
    ('epsilon_Q', r'$\epsilon_Q$  (process noise)'),
    ('epsilon_R', r'$\epsilon_R$  (obs noise)'),
]

# ── Collect iter-0 (pre-EM) errors for all seeds and all metrics ───────────────
iter0_nmf  = {k: [] for k, _ in METRICS_41C}
iter0_rand = {k: [] for k, _ in METRICS_41C}

for i, sk in enumerate(seed_keys_41):
    key_data, key_rand_init = jr.split(sk, 2)
    ctds_sp, tp_sp, obs_sp, _ = generate_ground_truth_and_data(
        key_data, T=T_DEFAULT, B=B_DEFAULT,
        D=D_DEFAULT, N_E=N_E_DEFAULT, N_I=N_I_DEFAULT
    )
    nmf_ip_sp  = get_nmf_init(ctds_sp, obs_sp)
    rand_ip_sp = get_random_init(ctds_sp, tp_sp,obs_sp, key_rand_init)
    errs_nmf_sp  = align_and_compute_errors(tp_sp, nmf_ip_sp)
    errs_rand_sp = align_and_compute_errors(tp_sp, rand_ip_sp)
    for k, _ in METRICS_41C:
        iter0_nmf[k].append(errs_nmf_sp[k])
        iter0_rand[k].append(errs_rand_sp[k])

iter0_nmf  = {k: np.array(v) for k, v in iter0_nmf.items()}
iter0_rand = {k: np.array(v) for k, v in iter0_rand.items()}

# ── Extract checkpoint errors for all metrics ──────────────────────────────────
def extract_checkpoint_metric(results_list, checkpoints, metric_key):
    mat = np.zeros((len(results_list), len(checkpoints)))
    for i, res in enumerate(results_list):
        for j, ck in enumerate(checkpoints):
            mat[i, j] = res['checkpoint_errors'][ck][metric_key]
    return mat

ck_errors_nmf  = {k: extract_checkpoint_metric(results_41_nmf,  ITER_CHECKPOINTS, k)
                  for k, _ in METRICS_41C}
ck_errors_rand = {k: extract_checkpoint_metric(results_41_rand, ITER_CHECKPOINTS, k)
                  for k, _ in METRICS_41C}

# Keep backwards-compat names used downstream
eA_nmf_iter0  = iter0_nmf['epsilon_A']
eA_rand_iter0 = iter0_rand['epsilon_A']
ck_eA_nmf     = ck_errors_nmf['epsilon_A']
ck_eA_rand    = ck_errors_rand['epsilon_A']

full_iters = np.array([0] + ITER_CHECKPOINTS)  # [0, 10, 25, 50, 100, 500,1000]

# Backwards-compat speed arrays for summary table
nmf_speed  = np.array([np.median(eA_nmf_iter0)]  + [np.median(ck_eA_nmf[:,  j]) for j in range(len(ITER_CHECKPOINTS))])
rand_speed = np.array([np.median(eA_rand_iter0)] + [np.median(ck_eA_rand[:, j]) for j in range(len(ITER_CHECKPOINTS))])

print("\nExperiment 4.1 convergence speed complete.")
for key, label in METRICS_41C:
    i0_nmf_m  = np.median(iter0_nmf[key])
    i0_rand_m = np.median(iter0_rand[key])
    fin_nmf_m  = np.median(ck_errors_nmf[key][:,  -1])
    fin_rand_m = np.median(ck_errors_rand[key][:, -1])
    print(f"  {label}: NMF {i0_nmf_m:.4f}→{fin_nmf_m:.4f}  |  "
          f"Rand {i0_rand_m:.4f}→{fin_rand_m:.4f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (key, label) in zip(axes.flat, METRICS_41C):
    i0_nmf  = iter0_nmf[key]
    i0_rand = iter0_rand[key]
    ck_nmf  = ck_errors_nmf[key]
    ck_rand = ck_errors_rand[key]

    nmf_med  = np.array([np.median(i0_nmf)]  + [np.median(ck_nmf[:,  j]) for j in range(len(ITER_CHECKPOINTS))])
    rand_med = np.array([np.median(i0_rand)] + [np.median(ck_rand[:, j]) for j in range(len(ITER_CHECKPOINTS))])
    nmf_lo   = np.array([np.percentile(i0_nmf,  25)] + [np.percentile(ck_nmf[:,  j], 25) for j in range(len(ITER_CHECKPOINTS))])
    nmf_hi   = np.array([np.percentile(i0_nmf,  75)] + [np.percentile(ck_nmf[:,  j], 75) for j in range(len(ITER_CHECKPOINTS))])
    rand_lo  = np.array([np.percentile(i0_rand, 25)] + [np.percentile(ck_rand[:, j], 25) for j in range(len(ITER_CHECKPOINTS))])
    rand_hi  = np.array([np.percentile(i0_rand, 75)] + [np.percentile(ck_rand[:, j], 75) for j in range(len(ITER_CHECKPOINTS))])

    ax.plot(full_iters, nmf_med, color=PALETTE[0], linewidth=2.0, linestyle='-',
            marker='o', markersize=5, label=f'NMF (median)')
    ax.fill_between(full_iters, nmf_lo, nmf_hi, color=PALETTE[0], alpha=0.18, label='NMF IQR')

    ax.plot(full_iters, rand_med, color='#555555', linewidth=2.0, linestyle='--',
            marker='s', markersize=5, label=f'Random (median)')
    ax.fill_between(full_iters, rand_lo, rand_hi, color='#555555', alpha=0.12, label='Random IQR')

    ax.set_xscale('log')
    ax.set_xticks(full_iters[1:])
    ax.set_xticklabels([str(x) for x in full_iters[1:]])
    ax.set_xlabel('EM Iteration (log scale)')
    ax.set_ylabel(f'Median {label}')
    ax.set_title(label)
    ax.legend(fontsize=8)

    # Annotate pre-EM starting points
    ax.annotate(f'NMF\n({nmf_med[0]:.3f})',
                xy=(full_iters[1], nmf_med[1]),
                xytext=(full_iters[1]*1.4, nmf_med[1]*1.15),
                fontsize=7, color=PALETTE[0],
                arrowprops=dict(arrowstyle='->', color=PALETTE[0], lw=0.8))
    ax.annotate(f'Rand\n({rand_med[0]:.3f})',
                xy=(full_iters[1], rand_med[1]),
                xytext=(full_iters[1]*1.4, rand_med[1]*0.85),
                fontsize=7, color='#555555',
                arrowprops=dict(arrowstyle='->', color='#555555', lw=0.8))

fig.suptitle(
    f'Experiment 4.1c: Convergence Speed — NMF vs. Random, All Metrics\n'
    f'(Shaded band = IQR across {NUM_SEEDS} seeds)',
    fontsize=13
)
save_figure(fig, 'exp4_1c_convergence_speed_all_metrics')
plt.show()

---
## Section 3: Experiment 4.2 — Initialization Quality vs. Data Volume

**Question:** Does the advantage of NMF initialization decrease as more data is available?

**Hypothesis:** With very large T·B, random initialization eventually finds solutions of comparable quality — the objective landscape becomes better-conditioned and local optima converge to the global one. With limited data, NMF initialization is critical. If the gap does not shrink with T, it implies NMF captures structural information (the signed block structure of A and C) that is not available from raw data volume alone — a noteworthy finding.

**Protocol:**
- Vary T ∈ {50, 100, 500, 1000}, B=5 fixed.
- 10 seeds × 2 strategies per condition.
- Compare final ε_A (200 iterations) at each T.
- Plot the gap: ε_A(random) − ε_A(NMF) vs. T.

In [ ]:
T_VALUES_42 = [50, 100, 500, 1000]
B_FIXED_42  = 5

seed_keys_42 = jr.split(jr.PRNGKey(42), NUM_SEEDS)

# results_42[T_val] = {'nmf': [err_dict, ...], 'rand': [err_dict, ...]}
results_42 = {}

print("Experiment 4.2: Initialization quality vs. data volume T")
print(f"  T values: {T_VALUES_42}, B={B_FIXED_42}, {NUM_SEEDS} seeds/condition")

for T_val in T_VALUES_42:
    nmf_errs_list  = []
    rand_errs_list = []

    for i, sk in enumerate(seed_keys_42):
        key_data, key_rand_init = jr.split(jr.fold_in(sk, T_val), 2)

        # Generate ground truth and data at this T
        ctds_42, tp_42, obs_42, _ = generate_ground_truth_and_data(
            key_data,
            T=T_val, B=B_FIXED_42,
            D=D_DEFAULT, N_E=N_E_DEFAULT, N_I=N_I_DEFAULT
        )

        # NMF
        nmf_ip_42 = get_nmf_init(ctds_42, obs_42)
        res_nmf_42 = run_em_manual(
            ctds_42, nmf_ip_42, obs_42, tp_42,
            num_iters=NUM_EM_ITERS, checkpoints=[NUM_EM_ITERS]
        )
        nmf_errs_list.append(res_nmf_42['final_errors'])

        # Random
        rand_ip_42 = get_random_init(ctds_42,tp_42, obs_42, key_rand_init)
        res_rand_42 = run_em_manual(
            ctds_42, rand_ip_42, obs_42, tp_42,
            num_iters=NUM_EM_ITERS, checkpoints=[NUM_EM_ITERS]
        )
        rand_errs_list.append(res_rand_42['final_errors'])

        print(f"  T={T_val:5d}, seed {i+1:2d}/{NUM_SEEDS}: "
              f"NMF ε_A={res_nmf_42['final_errors']['epsilon_A']:.4f}, "
              f"RAND ε_A={res_rand_42['final_errors']['epsilon_A']:.4f}",
              flush=True)

    results_42[T_val] = {'nmf': nmf_errs_list, 'rand': rand_errs_list}

print("\nExperiment 4.2 data collection complete.")

In [ ]:
# ── Aggregate and plot 4.2 — all four metrics ──────────────────────────────────

METRICS_42 = [
    ('epsilon_A', r'$\epsilon_A$  (dynamics)'),
    ('epsilon_C', r'$\epsilon_C$  (emissions)'),
    ('epsilon_Q', r'$\epsilon_Q$  (process noise)'),
    ('epsilon_R', r'$\epsilon_R$  (obs noise)'),
]

# Build aggregated stats for every metric
agg_42 = {}   # metric_key → dict of arrays
for key, _ in METRICS_42:
    nmf_means, rand_means, gap_means, gap_ses = [], [], [], []
    nmf_ses, rand_ses = [], []
    for T_val in T_VALUES_42:
        nmf_v  = np.array([e[key] for e in results_42[T_val]['nmf']])
        rand_v = np.array([e[key] for e in results_42[T_val]['rand']])
        gaps   = rand_v - nmf_v
        nmf_means.append(np.mean(nmf_v));   nmf_ses.append(np.std(nmf_v,  ddof=1) / np.sqrt(NUM_SEEDS))
        rand_means.append(np.mean(rand_v)); rand_ses.append(np.std(rand_v, ddof=1) / np.sqrt(NUM_SEEDS))
        gap_means.append(np.mean(gaps));    gap_ses.append(np.std(gaps,    ddof=1) / np.sqrt(NUM_SEEDS))
    agg_42[key] = {
        'nmf_m'  : np.array(nmf_means),  'nmf_se'  : np.array(nmf_ses),
        'rand_m' : np.array(rand_means), 'rand_se' : np.array(rand_ses),
        'gap_m'  : np.array(gap_means),  'gap_se'  : np.array(gap_ses),
    }

# Keep backwards-compat variables used in summary table / combined figure
gaps_mean_42 = agg_42['epsilon_A']['gap_m']
gaps_se_42   = agg_42['epsilon_A']['gap_se']
eA_nmf_mean_42  = agg_42['epsilon_A']['nmf_m'];  eA_nmf_se_42  = agg_42['epsilon_A']['nmf_se']
eA_rand_mean_42 = agg_42['epsilon_A']['rand_m']; eA_rand_se_42 = agg_42['epsilon_A']['rand_se']

# ── Plot 4.2.a — Recovery error vs. T and NMF gap, all metrics ────────────────
fig, axes = plt.subplots(len(METRICS_42), 2, figsize=(11, 4 * len(METRICS_42)))

for row, (key, label) in enumerate(METRICS_42):
    d = agg_42[key]

    # Left panel: raw error for both strategies
    ax = axes[row, 0]
    ax.errorbar(T_VALUES_42, d['nmf_m'],  yerr=d['nmf_se'],
                color=PALETTE[0], marker='o', linestyle='-',
                capsize=4, label='NMF init')
    ax.errorbar(T_VALUES_42, d['rand_m'], yerr=d['rand_se'],
                color='#555555', marker='s', linestyle='--',
                capsize=4, label='Random init')
    ax.set_xscale('log')
    ax.set_xticks(T_VALUES_42)
    ax.set_xticklabels([str(t) for t in T_VALUES_42])
    ax.set_xlabel('T (log scale)')
    ax.set_ylabel(f'Mean {label} at convergence')
    ax.set_title(f'{label} vs. T  (B={B_FIXED_42})')
    ax.legend(fontsize=9)

    # Right panel: gap ε(random) − ε(NMF)
    ax = axes[row, 1]
    ax.errorbar(T_VALUES_42, d['gap_m'], yerr=d['gap_se'],
                color=PALETTE[2], marker='D', linestyle='-',
                capsize=4, linewidth=2)
    ax.axhline(y=0, color='grey', linestyle='--', linewidth=1.2, label='No advantage')

    gap_shrinks_metric = d['gap_m'][-1] < d['gap_m'][0]
    if gap_shrinks_metric:
        ann, ann_color = 'Gap shrinks with T\n→ random catches up', 'steelblue'
    else:
        ann, ann_color = 'Gap does NOT shrink\n→ NMF captures structure', 'firebrick'
    ax.text(0.97, 0.95, ann, transform=ax.transAxes, ha='right', va='top',
            fontsize=8.5, color=ann_color,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

    ax.set_xscale('log')
    ax.set_xlabel('T (log scale)')
    ax.set_ylabel(f'Gap: {label}(rand) − {label}(NMF)')
    ax.set_title(f'NMF Advantage — {label}  (B={B_FIXED_42}, {NUM_SEEDS} seeds)')
    ax.legend(fontsize=9)

fig.suptitle(
    f'Experiment 4.2: NMF Advantage vs. Data Volume T — All Metrics\n'
    f'(B={B_FIXED_42}, {NUM_SEEDS} seeds, {NUM_EM_ITERS} EM iterations)',
    fontsize=13
)
save_figure(fig, 'exp4_2a_nmf_advantage_vs_T_all_metrics')
plt.show()

# ── Print summary ──────────────────────────────────────────────────────────────
print("\nExperiment 4.2 Summary — all metrics")
for key, label in METRICS_42:
    d = agg_42[key]
    shrinks = d['gap_m'][-1] < d['gap_m'][0]
    print(f"\n  {label}  (gap shrinks: {shrinks})")
    for T_val, gap, se in zip(T_VALUES_42, d['gap_m'], d['gap_se']):
        print(f"    T={T_val:5d}: gap = {gap:.4f} ± {se:.4f}")

gap_shrinks = gaps_mean_42[-1] < gaps_mean_42[0]   # ε_A, for back-compat

---
## Section 4: Experiment 4.3 — Quality of NMF Initialization Itself

**Question:** How good is the NMF initialization as a starting point, independently of EM?

**Hypothesis:** The NMF initialization produces parameters that are already substantially closer to the ground truth than random initialization — EM is needed to refine, but NMF does most of the work. Concretely:
- ε_A(NMF init) << ε_A(random init)
- ε_A(after EM from NMF) <= ε_A(NMF init)  but the further drop is smaller than the NMF → random gap

If NMF is already very close to the EM-converged solution, it implies EM is adding little and the NMF initialization effectively solves the problem on its own. That would be an important calibration of when EM is necessary.

**Protocol:** Three-stage comparison across 20 seeds — (1) random init before EM, (2) NMF init before EM, (3) EM-converged from NMF init. Bar plot with ε_C and ε_A.

**Why ε_C for the main panel:** ε_C is more directly controlled by the NMF initialization (the NMF factorization directly estimates C), so the three-stage comparison is sharpest for this metric.

In [ ]:
NUM_SEEDS_43 = 20   # more seeds for a tighter bar plot
seed_keys_43 = jr.split(jr.PRNGKey(43), NUM_SEEDS_43)

# Storage: three stages × two metrics (ε_A, ε_C)
stage1_rand = []   # (1) random init, before EM
stage2_nmf  = []   # (2) NMF init, before EM
stage3_em   = []   # (3) EM-converged from NMF init

print(f"Experiment 4.3: Three-stage comparison ({NUM_SEEDS_43} seeds)")
print("  Stage 1: random init (before EM)")
print("  Stage 2: NMF init (before EM)")
print("  Stage 3: EM-converged from NMF init")

for i, sk in enumerate(seed_keys_43):
    key_data, key_rand_init = jr.split(sk, 2)

    # ── Generate data ──────────────────────────────────────────────────────
    ctds_43, tp_43, obs_43, _ = generate_ground_truth_and_data(
        key_data,
        T=T_DEFAULT, B=B_DEFAULT,
        D=D_DEFAULT, N_E=N_E_DEFAULT, N_I=N_I_DEFAULT
    )

    # ── Stage 1: random init, before EM ───────────────────────────────────
    rand_ip_43 = get_random_init(ctds_43,tp_43, obs_43, key_rand_init)
    errs_rand  = align_and_compute_errors(tp_43, rand_ip_43)
    stage1_rand.append(errs_rand)

    # ── Stage 2: NMF init, before EM ──────────────────────────────────────
    nmf_ip_43 = get_nmf_init(ctds_43, obs_43)
    errs_nmf  = align_and_compute_errors(tp_43, nmf_ip_43)
    stage2_nmf.append(errs_nmf)

    # ── Stage 3: EM-converged from NMF init ───────────────────────────────
    res_em_43 = run_em_manual(
        ctds_43, nmf_ip_43, obs_43, tp_43,
        num_iters=NUM_EM_ITERS, checkpoints=[NUM_EM_ITERS]
    )
    stage3_em.append(res_em_43['final_errors'])

    print(f"  Seed {i+1:2d}/{NUM_SEEDS_43}: "
          f"rand ε_C={errs_rand['epsilon_C']:.4f} | "
          f"NMF ε_C={errs_nmf['epsilon_C']:.4f} | "
          f"EM ε_C={res_em_43['final_errors']['epsilon_C']:.4f}",
          flush=True)

print("\nExperiment 4.3 data collection complete.")

In [ ]:
# ── Aggregate 4.3 ──────────────────────────────────────────────────────────────

def agg_stage(stage_list, metric):
    vals = np.array([e[metric] for e in stage_list])
    return np.mean(vals), np.std(vals, ddof=1) / np.sqrt(len(vals)), vals

# ε_C
eC_rand_m,  eC_rand_se,  eC_rand_vals  = agg_stage(stage1_rand, 'epsilon_C')
eC_nmf_m,   eC_nmf_se,   eC_nmf_vals   = agg_stage(stage2_nmf,  'epsilon_C')
eC_em_m,    eC_em_se,    eC_em_vals     = agg_stage(stage3_em,   'epsilon_C')

# ε_A
eA_rand_m,  eA_rand_se,  eA_rand_vals  = agg_stage(stage1_rand, 'epsilon_A')
eA_nmf_m,   eA_nmf_se,   eA_nmf_vals   = agg_stage(stage2_nmf,  'epsilon_A')
eA_em_m,    eA_em_se,    eA_em_vals     = agg_stage(stage3_em,   'epsilon_A')

# Fraction of final error already achieved at NMF init
frac_C_done = (eC_rand_m - eC_nmf_m) / (eC_rand_m - eC_em_m + 1e-12)
frac_A_done = (eA_rand_m - eA_nmf_m) / (eA_rand_m - eA_em_m + 1e-12)

print("Experiment 4.3 — Three-stage comparison summary")
print(f"  Stage 1 (random init): ε_C={eC_rand_m:.4f}, ε_A={eA_rand_m:.4f}")
print(f"  Stage 2 (NMF init):    ε_C={eC_nmf_m:.4f}, ε_A={eA_nmf_m:.4f}")
print(f"  Stage 3 (EM from NMF): ε_C={eC_em_m:.4f}, ε_A={eA_em_m:.4f}")
print(f"\n  Fraction of improvement (random→EM) already done by NMF init:")
print(f"    ε_C: {frac_C_done*100:.1f}%")
print(f"    ε_A: {frac_A_done*100:.1f}%")

if frac_C_done > 0.7:
    print("\n  INTERPRETATION: NMF init achieves >70% of the improvement — EM adds only a small refinement.")
else:
    print("\n  INTERPRETATION: EM adds substantial improvement beyond the NMF starting point.")

# Assertions
for stage_mean_nmf, stage_mean_rand, metric_name in [
    (eC_nmf_m, eC_rand_m, 'epsilon_C'),
    (eA_nmf_m, eA_rand_m, 'epsilon_A'),
]:
    try:
        assert stage_mean_nmf < stage_mean_rand, \
            f"NMF init {metric_name}={stage_mean_nmf:.4f} >= random {stage_mean_rand:.4f}"
        print(f"  ASSERTION PASSED: NMF init has lower {metric_name} than random.")
    except AssertionError as e:
        print(f"  WARNING: {e}")

In [ ]:
# ── Plot 4.3.a — Error at three stages (main panel) ───────────────────────────
#
# Two-panel figure: left = ε_C (primary, NMF directly estimates C via NMF);
# right = ε_A (secondary, shows the indirect benefit on dynamics recovery).
# Each panel: bar chart + individual-point scatter for transparency.

stage_labels = [
    '(1) Random init\n(before EM)',
    '(2) NMF init\n(before EM)',
    '(3) EM from\nNMF init',
]
bar_colours_43 = [PALETTE[7], PALETTE[1], PALETTE[0]]  # grey, orange, blue

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, (means, ses, vals_list, metric_label, stage_vals) in zip(
    axes,
    [
        ([eC_rand_m, eC_nmf_m, eC_em_m],
         [eC_rand_se, eC_nmf_se, eC_em_se],
         [eC_rand_vals, eC_nmf_vals, eC_em_vals],
         r'$\epsilon_C$ (emission matrix)',
         [eC_rand_vals, eC_nmf_vals, eC_em_vals]),
        ([eA_rand_m, eA_nmf_m, eA_em_m],
         [eA_rand_se, eA_nmf_se, eA_em_se],
         [eA_rand_vals, eA_nmf_vals, eA_em_vals],
         r'$\epsilon_A$ (dynamics matrix)',
         [eA_rand_vals, eA_nmf_vals, eA_em_vals]),
    ]
):
    x_pos = np.arange(3)
    bars = ax.bar(
        x_pos, means, yerr=ses,
        color=bar_colours_43, capsize=5,
        edgecolor='white', linewidth=0.5, alpha=0.75,
        error_kw=dict(elinewidth=1.5, ecolor='black')
    )

    # Scatter individual seed values
    rng = np.random.default_rng(7)
    for x, vals in zip(x_pos, stage_vals):
        jitter = rng.uniform(-0.12, 0.12, size=len(vals))
        ax.scatter(x + jitter, vals, color='black', s=16, alpha=0.55, zorder=4)

    # Annotate bar heights
    for x, m, s in zip(x_pos, means, ses):
        ax.text(x, m + s + max(means) * 0.02, f'{m:.3f}',
                ha='center', fontsize=8.5, fontweight='bold')

    # Arrow showing the two-step improvement
    y_arrow = max(means) * 1.18
    ax.annotate('', xy=(1, means[1] + ses[1] * 0.5),
                xytext=(0, means[0] + ses[0] * 0.5),
                arrowprops=dict(arrowstyle='->', color='grey', lw=1.2))
    ax.annotate('', xy=(2, means[2] + ses[2] * 0.5),
                xytext=(1, means[1] + ses[1] * 0.5),
                arrowprops=dict(arrowstyle='->', color=PALETTE[0], lw=1.2))

    ax.set_xticks(x_pos)
    ax.set_xticklabels(stage_labels, fontsize=9)
    ax.set_ylabel('Normalised Frobenius Error')
    ax.set_title(metric_label)
    ax.set_ylim(0, max(means) * 1.45)

    # Legend patches
    legend_patches = [
        mpatches.Patch(color=bar_colours_43[0], alpha=0.75, label='Random init'),
        mpatches.Patch(color=bar_colours_43[1], alpha=0.75, label='NMF init (pre-EM)'),
        mpatches.Patch(color=bar_colours_43[2], alpha=0.75, label='EM from NMF'),
    ]
    ax.legend(handles=legend_patches, fontsize=8, loc='upper right')

fig.suptitle(
    f'Experiment 4.3a: Parameter Error at Three Stages\n'
    f'({NUM_SEEDS_43} seeds, {NUM_EM_ITERS} EM iterations)',
    fontsize=13
)
save_figure(fig, 'exp4_3a_three_stage_error')
plt.show()

print(f"\n  NMF init achieves {frac_C_done*100:.1f}% of the total ε_C reduction (random → EM).")
print(f"  NMF init achieves {frac_A_done*100:.1f}% of the total ε_A reduction (random → EM).")

In [ ]:
# ── Supplementary: paired scatter — NMF init ε vs. EM-converged ε ─────────────
#
# Each point is one seed. x-axis = ε at NMF init (stage 2); y-axis = ε after EM (stage 3).
# Points below the identity line mean EM improved on the NMF init.
# Points close to the identity line mean EM added little.

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

for ax, (xvals, yvals, metric_label) in zip(
    axes,
    [
        (eC_nmf_vals, eC_em_vals, r'$\epsilon_C$'),
        (eA_nmf_vals, eA_em_vals, r'$\epsilon_A$'),
    ]
):
    ax.scatter(xvals, yvals, color=PALETTE[0], s=40, alpha=0.75, zorder=3)

    # Identity line
    xy_max = max(np.max(xvals), np.max(yvals)) * 1.05
    xy_min = min(np.min(xvals), np.min(yvals)) * 0.95
    ax.plot([xy_min, xy_max], [xy_min, xy_max],
            'k--', linewidth=1.2, alpha=0.5, label='identity (no EM change)')

    # Fraction of points below identity
    frac_improved = np.mean(np.array(yvals) < np.array(xvals))
    ax.text(0.05, 0.95,
            f'{frac_improved*100:.0f}% of seeds improved by EM',
            transform=ax.transAxes, ha='left', va='top',
            fontsize=8.5, color='steelblue',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

    ax.set_xlabel(f'{metric_label} at NMF init (before EM)')
    ax.set_ylabel(f'{metric_label} after EM from NMF init')
    ax.set_title(f'Paired: NMF init vs. EM outcome ({metric_label})')
    ax.legend(fontsize=8)

fig.suptitle(
    'Experiment 4.3 Supplementary: Does EM Add Value Beyond NMF Init?\n'
    'Points below the diagonal = EM improved on the starting point.',
    fontsize=11
)
save_figure(fig, 'exp4_3_supplementary_paired_scatter')
plt.show()

---
## Section 5: Experiment Group 4 — Summary

All three experiments collected into a single summary table and combined figure.

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
print("=" * 68)
print("EXPERIMENT GROUP 4: INITIALIZATION QUALITY — SUMMARY")
print("=" * 68)

print(f"\n  Exp 4.1  NMF vs. Random at convergence ({NUM_EM_ITERS} iters, {NUM_SEEDS} seeds):")
print(f"           NMF  median ε_A = {np.median(eA_nmf_final):.4f}  ("
      f"σ = {np.std(eA_nmf_final):.4f})")
print(f"           RAND median ε_A = {np.median(eA_rand_final):.4f}  ("
      f"σ = {np.std(eA_rand_final):.4f})")
print(f"           NMF  median ε_C = {np.median(eC_nmf_final):.4f}")
print(f"           RAND median ε_C = {np.median(eC_rand_final):.4f}")
print(f"           NMF starts at ε_A = {nmf_speed[0]:.4f} (before EM)")
print(f"           RAND starts at ε_A = {rand_speed[0]:.4f} (before EM)")
print(f"           NMF advantage at iter 10: "
      f"{rand_speed[1] - nmf_speed[1]:.4f}")

print(f"\n  Exp 4.2  NMF advantage vs. T (B={B_FIXED_42}):")
for T_val, gap, se in zip(T_VALUES_42, gaps_mean_42, gaps_se_42):
    print(f"           T={T_val:5d}: gap = {gap:.4f} ± {se:.4f}")
print(f"           Gap shrinks with T: {gap_shrinks}")

print(f"\n  Exp 4.3  Three-stage comparison ({NUM_SEEDS_43} seeds):")
print(f"           Stage 1 (random init): ε_C={eC_rand_m:.4f}, ε_A={eA_rand_m:.4f}")
print(f"           Stage 2 (NMF init):    ε_C={eC_nmf_m:.4f}, ε_A={eA_nmf_m:.4f}")
print(f"           Stage 3 (EM from NMF): ε_C={eC_em_m:.4f}, ε_A={eA_em_m:.4f}")
print(f"           NMF accounts for {frac_C_done*100:.1f}% of ε_C reduction (random→EM)")
print(f"           NMF accounts for {frac_A_done*100:.1f}% of ε_A reduction (random→EM)")
print("=" * 68)

In [ ]:
# ── Combined summary figure (3 key panels) ─────────────────────────────────────
#
# Panel 1 (4.1a abridged): LL traces, both strategies, with medians.
# Panel 2 (4.2a): NMF advantage gap vs. T.
# Panel 3 (4.3a abridged): Three-stage ε_C bar chart.

fig = plt.figure(figsize=(15, 4.5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.38)

# ── Panel 1: LL traces ─────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
for trace in ll_rand_mat:
    ax1.plot(iters_axis, trace, **GRAY_THIN)
for trace in ll_nmf_mat:
    ax1.plot(iters_axis, trace, color=PALETTE[0], linewidth=0.7, alpha=0.5)
ax1.plot(iters_axis, ll_rand_med, color='#555555', linewidth=2.0, linestyle='--',
         label='Random (median)')
ax1.plot(iters_axis, ll_nmf_med,  color=PALETTE[0], linewidth=2.0, linestyle='-',
         label='NMF (median)')
ax1.set_xlabel('EM Iteration', fontsize=9)
ax1.set_ylabel('Marginal LL', fontsize=9)
ax1.set_title('4.1: LL Traces', fontsize=11)
ax1.legend(fontsize=8)
ax1.tick_params(labelsize=8)

# ── Panel 2: NMF advantage gap vs. T ──────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.errorbar(T_VALUES_42, gaps_mean_42, yerr=gaps_se_42,
             color=PALETTE[2], marker='D', linestyle='-',
             capsize=4, linewidth=2)
ax2.axhline(y=0, color='grey', linestyle='--', linewidth=1.0, alpha=0.7)
ax2.set_xscale('log')
ax2.set_xlabel('T (log scale)', fontsize=9)
ax2.set_ylabel(r'$\epsilon_A(\mathrm{rand}) - \epsilon_A(\mathrm{NMF})$', fontsize=9)
ax2.set_title('4.2: NMF Advantage vs. T', fontsize=11)
ax2.tick_params(labelsize=8)
gap_color = 'steelblue' if gap_shrinks else 'firebrick'
gap_text  = 'Gap shrinks' if gap_shrinks else 'Gap persists'
ax2.text(0.97, 0.95, gap_text, transform=ax2.transAxes,
         ha='right', va='top', fontsize=8, color=gap_color,
         bbox=dict(boxstyle='round,pad=0.2', facecolor='lightyellow', alpha=0.8))

# ── Panel 3: Three-stage ε_C bar chart ─────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
x_pos3 = np.arange(3)
means3  = [eC_rand_m,  eC_nmf_m,  eC_em_m]
ses3    = [eC_rand_se, eC_nmf_se, eC_em_se]
ax3.bar(x_pos3, means3, yerr=ses3,
        color=bar_colours_43, capsize=4, alpha=0.75,
        edgecolor='white', error_kw=dict(elinewidth=1.2))
for x, m, s in zip(x_pos3, means3, ses3):
    ax3.text(x, m + s + max(means3) * 0.03, f'{m:.3f}',
             ha='center', fontsize=8, fontweight='bold')
ax3.set_xticks(x_pos3)
ax3.set_xticklabels(['Random\ninit', 'NMF\ninit', 'EM from\nNMF'], fontsize=8)
ax3.set_ylabel(r'$\epsilon_C$', fontsize=9)
ax3.set_title('4.3: Three-Stage Comparison', fontsize=11)
ax3.tick_params(labelsize=8)
ax3.text(0.97, 0.95,
         f'NMF does\n{frac_C_done*100:.0f}% of work',
         transform=ax3.transAxes, ha='right', va='top',
         fontsize=8.5, color='steelblue',
         bbox=dict(boxstyle='round,pad=0.2', facecolor='lightyellow', alpha=0.8))

fig.suptitle(
    'Experiment Group 4: Initialization Quality — Summary Figure',
    fontsize=13, y=1.02
)
save_figure(fig, 'experiment_group4_summary')
plt.show()

print("\nAll Group 4 figures saved. Experiment Group 4 complete.")